# Proof of Concept: SAR Predictors Integration (Sentinel-1 & ALOS PALSAR)

This notebook demonstrates the end-to-end integration and verification of SAR predictors in `luma_ge`:
1. **Sentinel-1 C-Band SAR** (`COPERNICUS/S1_GRD`):
   - Incidence-angle masking & swath-edge filtering
   - Speckle filtering (Focal Median / Mean in linear power domain / Lee filter)
   - Backscatter polarization bands: `S1_VV`, `S1_VH` and polarization ratio `S1_VV_VH`
2. **ALOS PALSAR / PALSAR-2 L-Band SAR** (`SAR_EPOCH` / `SAR` v1):
   - QA mask (layover, shadow, nodata) and DN > 0 filtering
   - JAXA calibration: $\gamma^0_{\text{dB}} = 10 \cdot \log_{10}(\text{DN}^2) - 83.0$
   - Multi-collection year routing and 2011-2014 mission gap nearest-year fallback
   - Backscatter polarization bands: `PALSAR_HH`, `PALSAR_HV` and ratio `PALSAR_HH_HV`
3. **Predictor Orchestration** via `PredictorCalculation`:
   - Automatic Step 1 / Module 1 date inheritance
   - Stacking multispectral composite + terrain + spectral indices + SAR predictors
4. **Interactive Map Visualization** (geemap / folium)

## 1. Imports and Earth Engine Initialization

In [ ]:
import ee
import geemap
from luma_ge.ee_config import ensure_ee_initialized
from luma_ge.sar import get_sar_data
from luma_ge.predictor import PredictorCalculation
from luma_ge.data_acquisition import Reflectance_Data

# Ensure Earth Engine is initialized
ensure_ee_initialized()
print("Earth Engine successfully initialized!")

## 2. Define Area of Interest (AOI) & Timeframe
We select an area in South Sumatra, Indonesia with diverse land cover (wetlands, peatlands, oil palm/rubber plantations, and built-up areas).

In [ ]:
# Define an AOI (e.g. South Sumatra, Indonesia)
aoi = ee.Geometry.Rectangle([104.65, -3.05, 104.85, -2.85])

# Target Date Range (e.g. 2020 annual window)
start_date = '2020-01-01'
end_date = '2020-12-31'

print(f"AOI defined: {aoi.coordinates().getInfo()}")
print(f"Date Range: {start_date} to {end_date}")

## 3. Sentinel-1 C-Band Backscatter Retrieval
Retrieve Sentinel-1 backscatter using `get_sar_data().get_sentinel1()`.

In [ ]:
sar_calc = get_sar_data()

# Retrieve Sentinel-1 composite
s1_image = sar_calc.get_sentinel1(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    orbit_pass='BOTH',
    reducer='median',
    speckle_filter='focal_median',
    filter_radius_m=30.0,
    include_ratio=True
)

print("Sentinel-1 Band Names:", s1_image.bandNames().getInfo())

# Compute sample statistics over AOI
s1_stats = s1_image.reduceRegion(ee.Reducer.mean(), aoi, 100).getInfo()
print("Sentinel-1 Regional Mean Values (dB):")
for k, v in s1_stats.items():
    print(f"  {k}: {v:.2f} dB")

## 4. ALOS PALSAR / PALSAR-2 L-Band Backscatter Retrieval
Retrieve ALOS PALSAR backscatter using `get_sar_data().get_palsar()`.
Notice how it performs QA masking, 16-bit DN calibration, and polarization ratio calculation.

In [ ]:
# Retrieve ALOS PALSAR composite (2020)
palsar_image = sar_calc.get_palsar(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    reducer='median',
    speckle_filter='focal_median',
    filter_radius_m=30.0,
    include_ratio=True,
    mask_layover_shadow=True,
    allow_nearest_year_fallback=True
)

print("PALSAR Band Names:", palsar_image.bandNames().getInfo())

# Compute sample statistics over AOI
palsar_stats = palsar_image.reduceRegion(ee.Reducer.mean(), aoi, 100).getInfo()
print("PALSAR Regional Mean Values (gamma0 dB):")
for k, v in palsar_stats.items():
    print(f"  {k}: {v:.2f} dB")

### 4.1 Test Nearest-Year Fallback for Mission Gap (2011–2014)
Verify that querying a year within the gap (e.g., 2012) smoothly clamps to the closest available year (2010 in ALOS-1) without crashing.

In [ ]:
palsar_gap_image = sar_calc.get_palsar(
    aoi=aoi,
    start_date='2012-01-01',
    end_date='2012-12-31',
    allow_nearest_year_fallback=True
)
print("2012 Fallback bands:", palsar_gap_image.bandNames().getInfo())

## 5. End-to-End Orchestration: Optical + Terrain + Spectral + SAR
Demonstrate how `PredictorCalculation.compute_predictors()` automatically inherits the date range from Module 1 (`collection`) and stacks all predictors seamlessly.

In [ ]:
# Step 1: Acquire Optical Collection (Module 1)
ref_data = Reflectance_Data()
collection, stats = ref_data.get_optical_data(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    optical_data='L8_SR',
    cloud_cover=40,
    compute_detailed_stats=False
)
composite = collection.median().clip(aoi)

print(f"Optical collection acquired with stamped dates: "
      f"{collection.get('query_start_date').getInfo()} to {collection.get('query_end_date').getInfo()}")

# Step 5: Compute Complete Predictor Stack (Module 5)
predictor_service = PredictorCalculation()

predictor_config = {
    "individual_predictors": {
        "elevation": True,
        "slope": True,
        "spectral_indices": ["NDVI", "NDWI", "mNDWI"],
        "sentinel1": True,    # S1_VV, S1_VH, S1_VV_VH
        "palsar": True        # PALSAR_HH, PALSAR_HV, PALSAR_HH_HV
    }
}

result = predictor_service.compute_predictors(
    composite=composite,
    aoi=aoi,
    predictor_config=predictor_config,
    collection=collection
)

print("Computation Success:", result["success"])
print(f"Total Predictor Bands Stacked: {result['total_bands']}")
print("Stacked Bands:", result["stacked_predictors"].bandNames().getInfo())
print("Predictor Summary:", result["predictor_info"])

## 6. Interactive Map Visualization with Geemap
Visualize the optical composite, Sentinel-1 radar composite, and PALSAR L-band radar composite on an interactive map.

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 11)

# 1. Optical False Color (NIR / SWIR1 / Red)
optical_vis = {'bands': ['NIR', 'SWIR1', 'Red'], 'min': 0, 'max': 0.4}
Map.addLayer(composite, optical_vis, 'Landsat-8 False Color')

# 2. Sentinel-1 False Color (VV / VH / VV-VH ratio)
s1_vis = {'bands': ['S1_VV', 'S1_VH', 'S1_VV_VH'], 'min': [-20, -25, 0], 'max': [0, -5, 15]}
Map.addLayer(s1_image, s1_vis, 'Sentinel-1 C-band (VV/VH/Ratio)')

# 3. ALOS PALSAR False Color (HH / HV / HH-HV ratio)
palsar_vis = {'bands': ['PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV'], 'min': [-20, -25, 0], 'max': [0, -5, 15]}
Map.addLayer(palsar_image, palsar_vis, 'ALOS PALSAR-2 L-band (HH/HV/Ratio)')

Map.addLayerControl()
Map